# TP2 — Parte 4.3: DANN, análisis de resultados

Este notebook cubre los puntos **4.3.2** (análisis de curvas y tabla de accuracy) y **4.3.3** (visualización del espacio latente antes y después de la adaptación) para el modelo DANN entrenado en `models/dann.py`.

**Política del notebook:** si los artefactos ya existen en `checkpoints/` se muestran directamente; si no, se entrena.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

CKPT_DIR              = Path('checkpoints')
TRAINING_CURVES_PNG   = CKPT_DIR / 'dann_training_curves.png'
ACCURACY_TABLE_TXT    = CKPT_DIR / 'dann_accuracy_table.txt'
TSNE_PNG              = CKPT_DIR / 'dann_tsne_comparison.png'
DANN_CKPT             = CKPT_DIR / 'best_dann.pt'
BASELINE_CKPT         = CKPT_DIR / 'best_dann_baseline.pt'
PROTONET_CKPT         = CKPT_DIR / 'best_protonet.pt'

def need_training() -> bool:
    """True if any required artefact is missing."""
    required = [TRAINING_CURVES_PNG, ACCURACY_TABLE_TXT, TSNE_PNG,
                DANN_CKPT, BASELINE_CKPT]
    missing = [p for p in required if not p.exists()]
    for p in missing:
        print(f'  missing: {p}')
    return len(missing) > 0

RETRAIN = need_training()
print(f'\nRETRAIN = {RETRAIN}')

In [ ]:
# Optional retraining block: only runs if artefacts are missing.
# Uses the exact same entrypoint as the standalone script so the saved
# PNG/TXT live in the canonical paths above.
if RETRAIN:
    import subprocess, sys
    print('Training DANN + baseline via models.dann (mode=both) …')
    cmd = [sys.executable, '-m', 'models.dann', '--mode', 'both']
    result = subprocess.run(cmd, check=True)
    assert TRAINING_CURVES_PNG.exists(), 'training curves PNG not produced'
    print('Training finished.')
else:
    print('All artefacts present — skipping training.')

## 4.3.2 — Análisis de resultados

### Curvas de entrenamiento

L_cls (clasificación source), L_dom (clasificación de dominio) y accuracy sobre MNIST-M, para **Baseline** (sin GRL) y **DANN** (con GRL).

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
ax.imshow(mpimg.imread(TRAINING_CURVES_PNG))
ax.axis('off')
ax.set_title(f'Source: {TRAINING_CURVES_PNG}', fontsize=10)
plt.show()

**Lectura de las curvas.**

- *L_cls (source):* desciende rápido en ambos modelos: el clasificador de tarea aprende MNIST con o sin GRL.
- *L_dom:* en el **baseline** (sin GRL) no existe → se grafica la propia DomLoss que produce el modelo en evaluación, pero no influye en sus parámetros. En **DANN**, conforme `λ` crece, L_dom se acerca a log(2) ≈ 0.693 (random binario), señal de que el encoder logra producir features que el discriminador no puede separar por dominio.
- *Accuracy MNIST-M:* la curva del **baseline** sube al principio (porque generaliza algo de MNIST) y eventualmente se estanca/decae a medida que el modelo *sobreajusta al dominio source*. **DANN** sigue ganando target acc después de ese punto: el costo adversarial le impide aprovechar features específicos de MNIST y lo fuerza a apoyarse en features compartidos entre dominios.

### Tabla de accuracy final

In [ ]:
print(ACCURACY_TABLE_TXT.read_text())

## 4.3.3 — Visualización del espacio latente: antes y después de la adaptación

t-SNE sobre embeddings de **MNIST** y **MNIST-M** en tres momentos:

1. **Antes del entrenamiento** — encoder pre-entrenado solo con MNIST (ProtoNet).
2. **Finetuning** — sin GRL, solo L_cls (baseline).
3. **DANN** — con GRL y L_dom.

Para cada momento dos paneles con los **mismos embeddings**: arriba coloreados por **dominio** (MNIST vs MNIST-M), abajo coloreados por **clase** (dígito 0–9).

In [ ]:
fig, ax = plt.subplots(figsize=(18, 12))
ax.imshow(mpimg.imread(TSNE_PNG))
ax.axis('off')
ax.set_title(f'Source: {TSNE_PNG}', fontsize=10)
plt.show()

**Lectura de las visualizaciones.**

- *Por dominio (fila superior):* en el **pretrained** (solo MNIST) los puntos de MNIST forman clusters y los de MNIST-M caen en regiones aparte — los dominios son visualmente separables. En **DANN** los puntos de los dos dominios se entremezclan: el encoder ya no codifica de qué dominio viene cada imagen.
- *Por clase (fila inferior):* lo deseable es que los embeddings se agrupen **por dígito** independientemente del dominio. En **DANN** los clusters por clase contienen puntos de ambos dominios (mismo color, marcadores distintos), lo que indica que el encoder aprendió features compartidos relevantes para la clasificación. En el **baseline** muchos clusters tienen un solo dominio, evidencia del overfitting al source que observamos en la curva de accuracy.
- Este resultado *cualitativo* es consistente con la mejora *cuantitativa* en target accuracy reportada en la tabla anterior: la mezcla de dominios en el espacio latente es la condición previa para que los clasificadores entrenados en MNIST funcionen sobre MNIST-M.